# Baseline DINOv2 — train lại reference (KHÔNG MoE)

Notebook chạy `train_baseline.py`: DINOv2-B → pool → proj → L2 → triplet/supcon.
Tách hẳn khỏi HyMS-Route để có số baseline trung thực (kỳ vọng ~84-86 R@1 trên CUB).

## 1. Cài đặt

In [ ]:
# Cài các thư viện còn thiếu (Kaggle/Colab cần Internet ON).
# torch/torchvision đã có sẵn trên Kaggle & Colab; phần dưới chỉ bổ sung cái thiếu.
!pip install -q pytorch-metric-learning
import torch; print('CUDA:', torch.cuda.is_available())

## 2. Lấy code (clone repo nếu chưa có)

In [ ]:
# ── Lấy code: tự clone repo nếu chưa có (Colab/máy mới); bỏ qua nếu đã có sẵn ──
import os, sys

REPO_URL = 'https://github.com/trong5nhan6/retrieval.git'
REPO_DIR = 'retrieval'      # tên thư mục sau khi clone

if os.path.exists('config.py'):                       # đang ở ngay trong repo
    PROJECT_ROOT = os.path.abspath('.')
elif os.path.exists(os.path.join('..', 'config.py')): # đang ở notebooks/ (clone local)
    PROJECT_ROOT = os.path.abspath('..')
else:                                                 # môi trường trống -> clone
    if not os.path.isdir(REPO_DIR):
        os.system(f'git clone {REPO_URL} {REPO_DIR}')
    PROJECT_ROOT = os.path.abspath(REPO_DIR)

os.chdir(PROJECT_ROOT)                # cwd = gốc project (để train.py chạy thẳng)
sys.path.insert(0, PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)

## 3. Chọn bộ data + tải từ Drive

In [ ]:
from config import HCFG

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CHỌN BỘ DATA — chỉ sửa duy nhất dòng này: 'cub' | 'cars' | 'inshop'       ║
DATASET = 'cub'
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── Link Google Drive cho từng bộ (.zip). Điền link rồi chọn DATASET tương ứng ─
DATASET_URLS = {
    'cub':    'https://drive.google.com/file/d/1bwP3KNX1Kaj-sPanx32KrCl4_NVM5m2P/view?usp=drive_link',
    'cars':   '',   # TODO: dán link Drive cho Cars196.zip
    'inshop': '',   # TODO: dán link Drive cho In-shop.zip
}

DATASETS_DIR = os.path.join(PROJECT_ROOT, 'datasets')
os.makedirs(DATASETS_DIR, exist_ok=True)

import re, zipfile

def _drive_id(url):
    m = re.search(r'/d/([A-Za-z0-9_-]+)', url) or re.search(r'[?&]id=([A-Za-z0-9_-]+)', url)
    return m.group(1) if m else None

def fetch_dataset(name):
    """Tải <name>.zip từ Drive vào datasets/ rồi giải nén (bỏ qua nếu chưa có link)."""
    url = DATASET_URLS.get(name, '')
    if not url:
        print(f'[{name}] chưa có link Drive — bỏ qua (hãy điền DATASET_URLS).')
        return
    import gdown
    zip_path = os.path.join(DATASETS_DIR, f'{name}.zip')
    if not os.path.exists(zip_path):
        print(f'[{name}] tải từ Drive...')
        gdown.download(id=_drive_id(url), output=zip_path, quiet=False)
    print(f'[{name}] giải nén -> {DATASETS_DIR}')
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATASETS_DIR)
    print(f'[{name}] xong.')

# Tải đúng bộ đang chọn
fetch_dataset(DATASET)

# ── Trỏ HCFG tới datasets/ (đường dẫn tuyệt đối) ─────────────────────────────
HCFG.data_roots = {
    'cub':    os.path.join(DATASETS_DIR, 'CUB_200_2011'),
    'cars':   os.path.join(DATASETS_DIR, 'Cars196'),
    'inshop': os.path.join(DATASETS_DIR, 'In-shop Clothes Retrieval Benchmark'),
}
print('DATASET =', DATASET, '| root =', HCFG.data_roots[DATASET])

## 4. (Tuỳ chọn) override sampler dùng chung

In [ ]:
# ── (Tuỳ chọn) override các field HCFG DÙNG CHUNG (sampler/batch). ─────────────
# Các knob RIÊNG của baseline (pool, embed_dim, loss, epochs, finetune_blocks...)
# chỉnh ở cell "Cấu hình baseline" bên dưới, KHÔNG đặt ở đây.
OVERRIDES = {
    # 'classes_per_batch': 20,    # P lớp / batch
    # 'samples_per_class': 6,     # K ảnh / lớp
    # 'batch_size':        128,
}
import json
for k, v in OVERRIDES.items():
    if hasattr(HCFG, k): setattr(HCFG, k, v)
    else: print('Bỏ qua key không hợp lệ:', k)
os.makedirs('results', exist_ok=True)
with open('results/_notebook_overrides.json', 'w', encoding='utf-8') as f:
    json.dump(OVERRIDES, f, ensure_ascii=False, indent=2)
print('Đã áp', len(OVERRIDES), 'override (dùng chung).')

## 5. Cấu hình baseline + Train

In [ ]:
# ── CẤU HÌNH BASELINE — sửa các biến rồi chạy ô này để train ──────────────────
POOL       = 'cls'      # 'cls' | 'mean' | 'cls_mean'  (cls_mean thường nhỉnh hơn)
EMBED_DIM  = 512        # chiều embedding; 0 = retrieve trên feature thô 768-d
LOSS       = 'triplet'  # 'triplet' | 'supcon'
MARGIN     = 0.1
EPOCHS     = 30
FROZEN     = 5          # epoch Stage-1 (đóng băng backbone). Nếu EMBED_DIM=0 -> đặt 0
FT_BLOCKS  = 2          # số block ViT mở băng ở Stage-2
EVAL_EVERY = 2
SEED       = 42

cmd = (f"python train_baseline.py --dataset {DATASET} --seed {SEED} "
       f"--pool {POOL} --embed_dim {EMBED_DIM} --loss {LOSS} --margin {MARGIN} "
       f"--epochs {EPOCHS} --frozen_epochs {FROZEN} --finetune_blocks {FT_BLOCKS} "
       f"--eval_every {EVAL_EVERY}")
print(cmd)
!{cmd}

## 6. Eval lại từ checkpoint

In [ ]:
# ── Eval lại từ checkpoint tốt nhất (chỉ kênh base) ───────────────────────────
import torch
from train_baseline import DinoBaseline
from data.dml_dataset import get_dml_loaders
from eval.routerank import evaluate_self, evaluate_query_gallery

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
run = f"baseline_{DATASET}_seed{SEED}"
m = DinoBaseline(HCFG.vit_name, POOL, EMBED_DIM).to(dev)
ck = torch.load(f'results/checkpoints/best_{run}.pt', map_location=dev)
m.load_state_dict(ck['model']); m.eval()

L = get_dml_loaders(DATASET, HCFG)
rk = HCFG.recall_k_for(DATASET)
if DATASET == 'inshop':
    r = evaluate_query_gallery(m, L['query'], L['gallery'], dev, HCFG, use_routerank=False, recall_k=rk)
else:
    r = evaluate_self(m, L['test'], dev, HCFG, use_routerank=False, recall_k=rk)
print('BASE :', r['base'])
print('best ckpt @ epoch', ck['epoch'], '| R@1 =', ck['R@1'])

## 7. Xem biểu đồ

In [ ]:
# ── Xem biểu đồ đã auto-lưu trong results/ ───────────────────────────────────
from IPython.display import Image, display
run = f"baseline_{DATASET}_seed{SEED}"
for p in (f'results/plot_test_{run}.png', f'results/plot_train_{run}.png'):
    if os.path.exists(p):
        print(p); display(Image(p))
    else:
        print('(chưa có)', p)